# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df





In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04*.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock [mm]": 170.0,
    "rear_shock [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# ---- Batch orchestration ----

pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

print(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    print("  ", p.name)

sessions = []
events_all = []
metrics_all = []

for p in CSV_FILES:
    print(f"\nProcessing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
    )

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

print("\nDone.")
print(f"Sessions: {len(sessions)}")
print(f"Events rows: {len(events_df)}")
print(f"Metrics rows: {len(metrics_df)}")

print("CSV_FILES:", len(CSV_FILES))
print("About to loop...")

sessions = []
events_all = []
metrics_all = []

for i, p in enumerate(CSV_FILES):
    print(f"\n[{i}] Processing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
    )
    print("run_macro returned keys:", list(results.keys()))

    session = results["session"]
    sessions.append(session)
    print("sessions now:", len(sessions))

    if results.get("events") is not None:
        events_all.append(results["events"])
        print("events_all now:", len(events_all), "rows:", len(results["events"]))

    if results.get("metrics") is not None:
        metrics_all.append(results["metrics"])
        print("metrics_all now:", len(metrics_all), "rows:", len(results["metrics"]))

print("\nLoop done.")
print("Sessions:", len(sessions))
print("Events chunks:", len(events_all))
print("Metrics chunks:", len(metrics_all))


Found 2 files:
   2026-01-04_06-37-32.CSV
   2026-01-04_07-08-37.CSV

Processing 2026-01-04_06-37-32.CSV ...
time_s start/end: 0.002 283.292
median dt: 0.0020000000000024443
min dt: 0.0019999999999527063
max dt: 0.0020000000000095497
signals entries: 12
front_shock_dom_suspension [mm] -> {'kind': '', 'unit': 'mm', 'domain': 'suspension', 'op_chain': []}
front_shock_raw_dom_suspension [counts] -> {'kind': 'raw', 'unit': 'counts', 'domain': 'suspension', 'op_chain': []}
rear_shock_dom_suspension [mm] -> {'kind': '', 'unit': 'mm', 'domain': 'suspension', 'op_chain': []}
rear_shock_raw_dom_suspension [counts] -> {'kind': 'raw', 'unit': 'counts', 'domain': 'suspension', 'op_chain': []}
front_shock_dom_suspension [mm]_op_zeroed -> {'kind': '', 'unit': 'mm', 'domain': 'suspension', 'op_chain': ['zeroed']}
front_shock_norm [1] -> {'kind': '', 'unit': '1', 'domain': None, 'op_chain': []}
rear_shock_dom_suspension [mm]_op_zeroed -> {'kind': '', 'unit': 'mm', 'domain': 'suspension', 'op_chain': [